# 07 · Reachable junctions, checked against NetworkX

## What this checks

**Reachable junctions** finds the street junctions someone can reach from a starting point within a travel budget.
This check covers release 0.7.0.

Fieldwork says how it does this. The budget is the minutes multiplied by the speed. First it spends the walk from the
site to its nearest junction: a straight line in metres, rounded to the centimetre. Then it follows the cheapest paths
over the street lengths the network supplies. It keeps every junction where the walk plus the path is within the
budget. Each junction also carries the budget left on arrival, which the corridor step that follows uses.

The junctions reached decide the area a team treats as within reach. So it is worth checking.

## How it is checked

We repeat the traversal with **NetworkX**, a widely used Python library for networks. It is the right reference for a
reason. The case study this widget was taken from builds its walking areas with NetworkX. So we compare Fieldwork with
the program the original analysis used. Nothing here uses Fieldwork's code.

The walk onto the network is measured here too, not read from Fieldwork. We convert the site and its junction to
metres with **pyproj** in UTM zone 30 north, as Fieldwork declares, and take the straight line between them.

The data are the Soho street network of 738 junctions from the John Snow worked example. The settings are that
example's own: 5 minutes at 75 metres a minute, outbound, from the pump in Broadwick Street and the junction nearest it.

## What should happen

1. **The same junctions reached, exactly.** A junction either is or is not within the budget, so there is no
   tolerance to give.
2. **The walk onto the network** measured here, rounded to the centimetre, and counted in every cost.
3. **Every cost within 1 cm.** That is the rounding Fieldwork applies to the number it publishes, not an allowance for
   disagreement.
4. **Every budget left within 1 cm** of the budget less the cost. The corridor step that follows uses this number.

These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports, requested one at a time, so that a package the browser kernel lacks does not stop
# the others from loading. See 00-kernel-check for what this kernel really has.
NEEDED = ["networkx", "pyproj"]
try:
    import pyodide_js
except ImportError:
    pyodide_js = None
    print("Not a Pyodide kernel; using the ambient environment.")
if pyodide_js:
    for package in NEEDED:
        try:
            await pyodide_js.loadPackage(package)
            print("Loaded:", package)
        except Exception:
            print("Not in this kernel:", package)

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_07_reachability import run

report = run(Path.cwd() / "fixtures" / "reachability-soho-001.json")
print("criterion:", report["criterion"])
print("agrees:", report["agrees"])
print(f"network: {report['networkNodes']} junctions, {report['networkEdges']} directed streets")
print(f"budget: {report['budgetMetres']} m")
print(f"reached: Fieldwork {report['fieldwork']['reached']}, NetworkX {report['reference']['reached']}")
print(f"junctions only NetworkX reached: {report['reachedOnlyByReference'] or 'none'}")
print(f"junctions only Fieldwork reached: {report['reachedOnlyByFieldwork'] or 'none'}")
print(f"worst cost difference      {report['worstCostDifferenceM']:.3e} m")
print(f"worst remaining difference {report['worstRemainingDifferenceM']:.3e} m")

## The result on record

First recorded on 11 October 2026, against release 0.3.0. Rerun on 26 November 2026, against release 0.7.0:
**Agrees**.

The walk from the pump to its junction is 10.91 m, here and in Fieldwork. Both sides reached the same 134 of the 738
junctions. No junction was reached by only one side. The largest difference in cost was 5 mm, and so was the largest
difference in budget left.

At release 0.3.0 the walk was free, and both sides reached 142 junctions. Since release 0.4.0 Fieldwork spends the
walk first, so 6 junctions near the edge are no longer within reach. Until this check measured the walk too, it
disagreed on exactly those 6.

Running the cell above repeats the check on the same data.

## Where the 5 mm comes from

The 5 mm is not an error in either traversal. Fieldwork rounds the cost it reports to two decimal places of a metre.
So half a centimetre is the largest difference the published number can show. Both sides reach the same junctions,
in the same order, at the same costs. One of them publishes fewer digits.

That rounding still has an effect. The corridor step cuts a street at the share of its length the budget left
allows. So 5 mm of error in the budget left moves the end of a corridor by up to 5 mm. That is why the corridor
built in steps and the single Isochrone widget agree to 3.8 parts per million, not exactly.

## What this does not show

- That the street lengths are right. Both sides use the lengths the data state. Neither measures the streets. If the
  network is wrong, both are wrong together, and this check still passes.
- That the junctions reached are where anyone can walk in five minutes. Both sides share the 1854 walking speed. So
  this is agreement about arithmetic, not about walking.
- Travel towards the origin, or more than one origin, or another network. Only outbound travel from one origin on this
  network was tested.